<a href="https://colab.research.google.com/github/faisalepty/Sign-Language-CNN/blob/main/transfer_learningv2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [32]:
import torch
import torch.nn as nn
from torchvision import transforms, datasets
from torch.utils.data import ConcatDataset, DataLoader, random_split
import torchvision.models as models

# 1. Load pretrained model
model = models.efficientnet_b0(weights='IMAGENET1K_V1')

# 2. Freeze all layers (don't touch pretrained weights)
for param in model.parameters():
    param.requires_grad = False

# 3. Replace ONLY the final classifier for your 29 classes
model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(1280, 29)
)

# 4. Only the classifier trains, everything else is frozen
optimizer = torch.optim.AdamW(
    model.classifier.parameters(),  # only new layers
    lr=1e-3
)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
device = ("cuda" if torch.cuda.is_available() else "cpu")

In [33]:
model.load_state_dict(torch.load("/content/drive/MyDrive/asl-model/Copy of Copy of modelv2.pth", map_location=torch.device(device)))

<All keys matched successfully>

In [5]:
train_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.5],
                         [0.5])
])

val_transforms = transforms.Compose([
    # transforms.Grayscale(num_output_channels=1),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

In [6]:
import kagglehub

path = kagglehub.dataset_download("kapillondhe/american-sign-language", output_dir="./data")

print("Path to dataset files:", path)

100%|██████████| 4.64G/4.64G [05:33<00:00, 14.9MB/s]

Extracting files...


Path to dataset files: ./data


In [7]:
path1 = kagglehub.dataset_download("grassknoted/asl-alphabet", output_dir="./data1")

print("Path to dataset files:", path1)

Using Colab cache for faster access to the 'asl-alphabet' dataset.
Path to dataset files: /kaggle/input/asl-alphabet


In [8]:
path2 = kagglehub.dataset_download("debashishsau/aslamerican-sign-language-aplhabet-dataset")

print("Path to dataset files:", path2)

Using Colab cache for faster access to the 'aslamerican-sign-language-aplhabet-dataset' dataset.
Path to dataset files: /kaggle/input/aslamerican-sign-language-aplhabet-dataset


In [9]:
path3 = kagglehub.dataset_download("mrgeislinger/asl-rgb-depth-fingerspelling-spelling-it-out")

print("Path to dataset files:", path3)

Using Colab cache for faster access to the 'asl-rgb-depth-fingerspelling-spelling-it-out' dataset.
Path to dataset files: /kaggle/input/asl-rgb-depth-fingerspelling-spelling-it-out


In [10]:
path4 = kagglehub.dataset_download("ahmedkhanak1995/sign-language-gesture-images-dataset", output_dir="'/data4")

print("Path to dataset files:", path4)

100%|██████████| 191M/191M [00:14<00:00, 14.0MB/s]

Extracting files...


Path to dataset files: '/data4


In [11]:
path5 = kagglehub.dataset_download("alhasangamalmahmoud/american-sign-language-asl", output_dir="''/data5")

print("Path to dataset files:", path5)

100%|██████████| 2.09G/2.09G [02:26<00:00, 15.3MB/s]

Extracting files...


Path to dataset files: ''/data5


In [19]:
import shutil
import os

# Define the base source and destination directories
base_source_dir = "/content/''/data5/ASL"
base_destination_dir = '/content/data/ASL_Dataset/Train'

# Define the letters to iterate through
letters = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']

for letter in letters:
    # Construct source path with lowercase subfolder name
    source_dir = os.path.join(base_source_dir, letter.upper())
    # Destination path remains with uppercase letter
    destination_dir = os.path.join(base_destination_dir, letter.upper())

    # Create the parent directory for the destination if it doesn't exist
    os.makedirs(os.path.dirname(destination_dir), exist_ok=True)

    try:
        # Use copytree to copy the di
        shutil.copytree(source_dir, destination_dir, dirs_exist_ok=True)
        print(f"Successfully copied '{source_dir}' to '{destination_dir}'")
    except FileNotFoundError:
        print(f"Error: Source directory '{source_dir}' not found. Skipping {letter}")
    except Exception as e:
        print(f"An error occurred during copying '{source_dir}': {e}")


Successfully copied '/content/''/data5/ASL/A' to '/content/data/ASL_Dataset/Train/A'
Successfully copied '/content/''/data5/ASL/B' to '/content/data/ASL_Dataset/Train/B'
Successfully copied '/content/''/data5/ASL/C' to '/content/data/ASL_Dataset/Train/C'
Successfully copied '/content/''/data5/ASL/D' to '/content/data/ASL_Dataset/Train/D'
Successfully copied '/content/''/data5/ASL/E' to '/content/data/ASL_Dataset/Train/E'
Successfully copied '/content/''/data5/ASL/F' to '/content/data/ASL_Dataset/Train/F'
Successfully copied '/content/''/data5/ASL/G' to '/content/data/ASL_Dataset/Train/G'
Successfully copied '/content/''/data5/ASL/H' to '/content/data/ASL_Dataset/Train/H'
Successfully copied '/content/''/data5/ASL/I' to '/content/data/ASL_Dataset/Train/I'
Successfully copied '/content/''/data5/ASL/J' to '/content/data/ASL_Dataset/Train/J'
Successfully copied '/content/''/data5/ASL/K' to '/content/data/ASL_Dataset/Train/K'
Successfully copied '/content/''/data5/ASL/L' to '/content/data/A

In [21]:
import shutil
shutil.rmtree("/content/data/ASL_Dataset/Train/.ipynb_checkpoints", ignore_errors=True)

In [23]:
path = "data"
# Load datasets WITHOUT transforms initially
dataset1 = datasets.ImageFolder("/kaggle/input/aslamerican-sign-language-aplhabet-dataset/ASL_Alphabet_Dataset/asl_alphabet_train", transform=val_transforms)
dataset2 = datasets.ImageFolder("/content/data/ASL_Dataset/Train", transform=val_transforms)
dataset3 = datasets.ImageFolder("/kaggle/input/asl-alphabet/asl_alphabet_train/asl_alphabet_train", transform=val_transforms)


dataset = ConcatDataset([dataset1, dataset2, dataset3])

n_total = len(dataset)
n_train = int(0.8*n_total)
n_val = int(n_total - n_train)

train_dataset, val_dataset = random_split(dataset, [n_train, n_val], generator=torch.Generator().manual_seed(42))

In [24]:
print('Classes in dataset1:', dataset1.classes)
print('Classes in dataset2:', dataset2.classes)
print('Classes in dataset3:', dataset3.classes)

Classes in dataset1: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'del', 'nothing', 'space']
Classes in dataset2: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'del', 'nothing', 'space']
Classes in dataset3: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'del', 'nothing', 'space']


In [38]:
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, num_workers=2, pin_memory=True, persistent_workers=True)

In [ ]:
device = ("cuda" if torch.cuda.is_available() else "cpu")
# model = LeNet().to(device=device)
model = model.to(device)


In [27]:
from tqdm import tqdm

In [28]:
def train(model, train_loader):
  model.train()
  running_loss = 0.0
  for images, labels in tqdm(train_loader, desc="Training", leave=True):
      images, labels = images.to(device), labels.to(device)

      optimizer.zero_grad()
      outputs = model(images)
      loss = criterion(outputs, labels)
      loss.backward()
      optimizer.step()

      running_loss += loss.item()
  train_loss = running_loss / len(train_loader)
  return train_loss

In [29]:
def validate(model, val_loader):
  model.eval()
  val_loss = 0.0
  correct = 0

  total = 0
  with torch.no_grad():
      for images, labels in tqdm(val_loader, desc="validating", leave=True):
          images, labels = images.to(device), labels.to(device)
          outputs = model(images)
          loss = criterion(outputs, labels)
          val_loss += loss.item()
          preds = outputs.argmax(dim=1)
          correct += (preds == labels).sum().item()
          total += labels.size(0)
  val_loss /= len(val_loader)
  val_acc = correct / total
  return val_loss, val_acc

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [39]:
model.to(device)
epochs = 7
best_val_acc = 0.0
criterion = nn.CrossEntropyLoss()

for epoch in range(epochs):
    train_loss = train(model, train_loader)
    val_loss, val_acc = validate(model, val_loader)

    if val_acc > best_val_acc:
      best_val_acc = val_acc
      torch.save(model.state_dict(), f"/content/drive/My Drive/asl_modelv{epoch}.pth")



    print(f"Epoch [{epoch+1}/{epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

validating: 100%|██████████| 859/859 [08:37<00:00,  1.66it/s]


Epoch [1/7], Train Loss: 0.5132, Val Loss: 0.2568, Val Acc: 0.9270


validating: 100%|██████████| 859/859 [07:21<00:00,  1.95it/s]


Epoch [2/7], Train Loss: 0.5107, Val Loss: 0.2606, Val Acc: 0.9269


validating: 100%|██████████| 859/859 [07:30<00:00,  1.91it/s]


Epoch [3/7], Train Loss: 0.5101, Val Loss: 0.2618, Val Acc: 0.9248


Training:   2%|▏         | 73/3434 [00:37<28:33,  1.96it/s]


KeyboardInterrupt: 